In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/playground-series-s6e10/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e10/train.csv
/kaggle/input/competitions/playground-series-s6e10/test.csv


In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline 
from sklearn.impute import SimpleImputer

In [3]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e10/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e10/test.csv")
sample = pd.read_csv("/kaggle/input/competitions/playground-series-s6e10/sample_submission.csv")

train_cc = train.copy()
test_cc = test.copy()
sample_cc = sample.copy()

# Dataset Overview

In [4]:
print("Train dataset shape -- ", train.shape)
print("Test dataset shape -- ", test.shape)
print("Copied Train dataset shape -- ", train_cc.shape)
print("Copied Test dataset shape -- ", test_cc.shape)

Train dataset shape --  (699635, 23)
Test dataset shape --  (299844, 22)
Copied Train dataset shape --  (699635, 23)
Copied Test dataset shape --  (299844, 22)


In [5]:
train_cc.info()
print("="*50)
print("first 10 rows of the training dataset")
train_cc.head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 699635 entries, 0 to 699634
Data columns (total 23 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   id                                 699635 non-null  int64  
 1   Age                                699635 non-null  int64  
 2   Flight Distance                    699635 non-null  int64  
 3   Inflight wifi service              699635 non-null  int64  
 4   Departure/Arrival time convenient  699635 non-null  int64  
 5   Ease of Online booking             699635 non-null  int64  
 6   Gate location                      699635 non-null  int64  
 7   Food and drink                     699635 non-null  int64  
 8   Online boarding                    699635 non-null  int64  
 9   Seat comfort                       699635 non-null  int64  
 10  Inflight entertainment             699635 non-null  int64  
 11  On-board service                   6996

,id,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,...,Baggage handling,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,satisfaction
0,0,36,694,4,4,4,3,1,4,1,...,4,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
1,1,56,651,5,5,5,5,5,4,4,...,5,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,True
2,2,31,1371,3,4,2,5,3,3,3,...,4,5,3,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
3,3,10,1616,3,5,3,3,3,3,3,...,3,4,3,0,0.0,Male,Loyal Customer,Personal Travel,Eco,False
4,4,23,481,3,4,3,4,5,3,5,...,5,3,5,5,0.0,Female,disloyal Customer,Business travel,Eco,False
5,5,34,584,4,4,4,4,1,4,1,...,4,4,1,0,0.0,Female,disloyal Customer,Business travel,Business,False
6,6,36,1763,5,5,5,5,3,4,4,...,4,3,3,0,0.0,Female,Loyal Customer,Business travel,Business,True
7,7,57,861,1,3,1,3,4,4,4,...,5,4,2,0,0.0,Female,Loyal Customer,Personal Travel,Eco,False
8,8,60,581,3,3,3,3,5,4,5,...,4,4,5,0,0.0,Female,Loyal Customer,Business travel,Business,True
9,9,44,588,3,3,2,2,2,3,2,...,4,1,2,0,0.0,Female,disloyal Customer,Business travel,Eco,False


In [6]:
# datatype identification

datatype_summary = pd.DataFrame({
    "Feature": train_cc.columns,
    "Data type": train_cc.dtypes.values,
    "category": [
        "Categorical" if train[col].dtype == "object" else "Numerical"
        for col in train_cc.columns
    ]
})

display(datatype_summary)

,Feature,Data type,category
0,id,int64,Numerical
1,Age,int64,Numerical
2,Flight Distance,int64,Numerical
3,Inflight wifi service,int64,Numerical
4,Departure/Arrival time convenient,int64,Numerical
5,Ease of Online booking,int64,Numerical
6,Gate location,int64,Numerical
7,Food and drink,int64,Numerical
8,Online boarding,int64,Numerical
9,Seat comfort,int64,Numerical
